In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

df = pd.read_pickle("../Data/feature_database.pkl")
df = df[~df['Activity'].astype(str).str.contains('_')]

target_joint = 'G_DA_X_Target'
df = df.dropna(subset=[target_joint])

meta_cols = ['Subject', 'Timestamp', 'Activity', 'Gait', 'Reps']
gonio_cols = [col for col in df.columns if '_Target' in col]
feature_cols = [col for col in df.columns if col not in meta_cols + gonio_cols]

for sub in df['Subject'].unique():
    sub_mask = df['Subject'] == sub
    
    # Standardize features
    scaler = StandardScaler()
    df.loc[sub_mask, feature_cols] = scaler.fit_transform(df.loc[sub_mask, feature_cols])
    
    # Mean-center the target to remove goniometer calibration offsets
    target_mean = df.loc[sub_mask, target_joint].mean()
    df.loc[sub_mask, target_joint] = df.loc[sub_mask, target_joint] - target_mean

X = df[feature_cols].values
y = df[target_joint].values
groups = df['Subject'].values

print(f"Total features entering PCA: {len(feature_cols)}")
print(f"Regression Target: {target_joint} (Mean-Centered)")
print("-" * 50)

models = {
    "Ridge Regression": Ridge(alpha=10.0, random_state=42),
    "SVR (RBF)": SVR(kernel='rbf', C=10.0, epsilon=0.1),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, 
        max_depth=10, 
        min_samples_leaf=15, 
        random_state=42, 
        n_jobs=-1
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(
        max_depth=5, 
        min_samples_leaf=30, 
        random_state=42
    )
}

logo = LeaveOneGroupOut()

for model_name, reg in models.items():
    print(f"\nTesting: {model_name}")
    
    pipeline = Pipeline([
        ('pca', PCA(n_components=0.99, random_state=42)),
        ('regressor', reg)
    ])
    
    train_maes = []
    test_maes = []
    test_r2s = []
    
    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        test_subject = groups[test_idx[0]]
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        pipeline.fit(X_train, y_train)
        
        train_preds = pipeline.predict(X_train)
        test_preds = pipeline.predict(X_test)
        
        train_mae = mean_absolute_error(y_train, train_preds)
        test_mae = mean_absolute_error(y_test, test_preds)
        test_r2 = r2_score(y_test, test_preds)
        
        train_maes.append(train_mae)
        test_maes.append(test_mae)
        test_r2s.append(test_r2)
        
        print(f"Fold {fold+1} (Sub {test_subject}): Train MAE = {train_mae:.2f}° | Test MAE = {test_mae:.2f}° | Test R² = {test_r2:.3f}")

    print(f">> Average Train MAE: {np.mean(train_maes):.2f}° | Average Test MAE: {np.mean(test_maes):.2f}° | Average Test R²: {np.mean(test_r2s):.3f}")

Total features entering PCA: 78
Regression Target: G_DA_X_Target (Mean-Centered)
--------------------------------------------------

Testing: Ridge Regression
Fold 1 (Sub 01): Train MAE = 0.71° | Test MAE = 1.72° | Test R² = -1.108
Fold 2 (Sub 09): Train MAE = 0.50° | Test MAE = 2.10° | Test R² = -1.227
Fold 3 (Sub 17): Train MAE = 0.69° | Test MAE = 1.34° | Test R² = -0.166
>> Average Train MAE: 0.63° | Average Test MAE: 1.72° | Average Test R²: -0.834

Testing: SVR (RBF)
Fold 1 (Sub 01): Train MAE = 0.18° | Test MAE = 1.02° | Test R² = 0.271
Fold 2 (Sub 09): Train MAE = 0.18° | Test MAE = 2.05° | Test R² = -1.033
Fold 3 (Sub 17): Train MAE = 0.20° | Test MAE = 1.15° | Test R² = 0.089
>> Average Train MAE: 0.19° | Average Test MAE: 1.41° | Average Test R²: -0.224

Testing: Random Forest
Fold 1 (Sub 01): Train MAE = 0.41° | Test MAE = 1.01° | Test R² = 0.256
Fold 2 (Sub 09): Train MAE = 0.28° | Test MAE = 2.06° | Test R² = -0.917
Fold 3 (Sub 17): Train MAE = 0.36° | Test MAE = 1.11° | 